In [ ]:
import sys, json, pathlib
from html import escape
from collections import defaultdict
from IPython.display import display, HTML

here = pathlib.Path().resolve()
root = here
for _ in range(4):
    if (root / 'grammar_errors').is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from grammar_errors import ErrorChecker

PARTITION = root / 'processed_data' / 'sentences' / 'Student-2' / 'lesson-1' / 'paragraphs_custom.json'
partition = json.loads(PARTITION.read_text(encoding='utf-8'))

print(f"Student : {partition['student']}")
print(f"Lesson  : {partition['lesson']}")
print(f"Method  : {partition['partition_method']}")
print(f"Paragraphs: {len(partition['paragraphs'])}")

In [ ]:
checker = ErrorChecker()
print('LanguageTool ready.')

In [ ]:
# ── Run ErrorChecker on every paragraph ──────────────────────────────────────
# errors_by_para[para_id] = list of error records
errors_by_para = {}

for para in partition['paragraphs']:
    texts = [s['text'] for s in para['sentences']]
    start_idx = para['sentences'][0]['index']
    records = checker.check_sentences(texts, start_index=start_idx)
    errors_by_para[para['id']] = records
    print(f"P{para['id']:02d} [{len(texts):3d} sentences] {len(records):3d} errors — {para['label']}")

checker.close()

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

DIM_COLORS = {
    'A': {'bg': '#DBEAFE', 'border': '#3B82F6', 'badge': '#1D4ED8', 'label': 'Sentence Architecture'},
    'B': {'bg': '#FEF3C7', 'border': '#F59E0B', 'badge': '#92400E', 'label': 'Tense & Aspect Mastery'},
    'C': {'bg': '#D1FAE5', 'border': '#10B981', 'badge': '#065F46', 'label': 'Nominal Precision'},
    'D': {'bg': '#EDE9FE', 'border': '#8B5CF6', 'badge': '#4C1D95', 'label': 'Modal & Functional Range'},
}

SEV_LABEL = {1: 'low', 2: 'low', 3: 'medium', 4: 'high', 5: 'high'}
SEV_ALPHA = {'low': '44', 'medium': '88', 'high': 'cc'}  # hex alpha for bg


def _word_count(text: str) -> int:
    return len(text.split())


def compute_score(errors: list[dict], sentences: list[dict]) -> int:
    """0-100 quality score: 100 - weighted_errors / word_count * 100, clamped."""
    total_words = sum(_word_count(s['text']) for s in sentences)
    if total_words == 0:
        return 100
    weighted_sum = sum(e['weight'] for e in errors)
    raw = weighted_sum / total_words
    return max(0, round(100 * (1 - raw)))


def score_label(score: int) -> tuple[str, str]:
    """Return (label, hex_color) for a score."""
    if score >= 80:
        return 'Good', '#16a34a'
    if score >= 60:
        return 'Moderate', '#d97706'
    if score >= 40:
        return 'Weak', '#ea580c'
    return 'Poor', '#dc2626'


def sentence_to_html(sentence: str, errors: list[dict], uid_prefix: str) -> str:
    """Wrap error spans in a sentence with clickable tooltips."""
    # Sort and de-overlap (keep first match per character)
    errs = sorted(errors, key=lambda e: e['offset'])
    non_overlap, last_end = [], 0
    for e in errs:
        if e['offset'] >= last_end:
            non_overlap.append(e)
            last_end = e['offset'] + e['error_length']

    html, pos = '', 0
    for i, e in enumerate(non_overlap):
        html += escape(sentence[pos:e['offset']])
        dim = e['dimension_code']
        col = DIM_COLORS[dim]
        sev = SEV_LABEL[e['weight']]
        alpha = SEV_ALPHA[sev]
        bg = col['border'] + alpha
        border = col['border']
        uid = f"{uid_prefix}_e{i}"

        suggestions = ', '.join(e['replacements'][:3]) if e['replacements'] else 'none'
        tip_html = (
            f"<b>{escape(e['grammar_category'])}</b><br>"
            f"<span style='color:{col['badge']}'>{escape(e['dimension_label'])}</span><br>"
            f"{escape(e['message'])}<br>"
            f"<i>Suggestion:</i> {escape(suggestions)}<br>"
            f"<b>Severity:</b> {sev} &nbsp; <b>Weight:</b> {e['weight']}/5"
        )

        html += (
            f'<span id="{uid}" class="err-span" '
            f'style="background:{bg};border-bottom:2px solid {border};'
            f'border-radius:2px;padding:1px 2px;cursor:pointer;position:relative;" '
            f'onclick="toggleTip(\'{uid}\')" '
            f'title="{escape(e[\"grammar_category\"])} | {sev} | weight {e[\"weight\"]}">'
            f'{escape(sentence[e["offset"]:e["offset"]+e["error_length"]])}'
            f'<span id="tip_{uid}" class="err-tip" style="display:none;position:absolute;'
            f'bottom:125%;left:0;z-index:999;background:white;border:1.5px solid {border};'
            f'border-radius:6px;padding:8px 10px;min-width:220px;max-width:320px;'
            f'box-shadow:0 4px 12px rgba(0,0,0,.15);font-size:12px;line-height:1.5;'
            f'white-space:normal;text-align:left;">'
            f'{tip_html}</span></span>'
        )
        pos = e['offset'] + e['error_length']

    html += escape(sentence[pos:])
    return html

print('Helpers loaded.')

In [ ]:
# ── Build full HTML report ────────────────────────────────────────────────────

def render_report(partition: dict, errors_by_para: dict) -> None:
    js = """
    <script>
    function toggleTip(uid) {
        var tip = document.getElementById('tip_' + uid);
        if (!tip) return;
        var visible = tip.style.display !== 'none';
        // hide all open tips first
        document.querySelectorAll('.err-tip').forEach(function(t){ t.style.display='none'; });
        if (!visible) tip.style.display = 'block';
    }
    document.addEventListener('click', function(e){
        if (!e.target.closest('.err-span')) {
            document.querySelectorAll('.err-tip').forEach(function(t){ t.style.display='none'; });
        }
    });
    </script>
    """

    css = """
    <style>
    .para-card {
        font-family: Georgia, serif;
        font-size: 14px;
        line-height: 1.8;
        border: 1px solid #e2e8f0;
        border-radius: 8px;
        margin: 16px 0;
        overflow: hidden;
    }
    .para-header {
        background: #f8fafc;
        border-bottom: 1px solid #e2e8f0;
        padding: 8px 14px;
        font-family: sans-serif;
        font-size: 12px;
        color: #64748b;
        display: flex;
        justify-content: space-between;
        align-items: center;
    }
    .para-label { font-weight: 700; color: #1e293b; font-size: 13px; }
    .para-body  { padding: 12px 16px; }
    .para-footer {
        background: #f8fafc;
        border-top: 1px solid #e2e8f0;
        padding: 8px 14px;
        font-family: sans-serif;
        font-size: 12px;
        display: flex;
        align-items: center;
        gap: 16px;
    }
    .score-pill {
        font-weight: 800;
        font-size: 14px;
        padding: 2px 10px;
        border-radius: 12px;
        color: white;
    }
    .dim-legend {
        display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 10px;
        font-family: sans-serif; font-size: 11px;
    }
    .dim-chip {
        padding: 2px 8px;
        border-radius: 10px;
        border: 1.5px solid;
        font-weight: 600;
    }
    </style>
    """

    legend = '<div class="dim-legend">'
    for code, c in DIM_COLORS.items():
        legend += (f'<span class="dim-chip" style="background:{c["bg"]};'
                   f'border-color:{c["border"]};color:{c["badge"]}">'
                   f'{code} — {c["label"]}</span>')
    legend += '</div>'

    sev_legend = (
        '<div style="font-family:sans-serif;font-size:11px;color:#64748b;margin-bottom:14px;">'
        'Opacity = severity &nbsp;|&nbsp; '
        '<span style="opacity:.4">low (1–2)</span> &nbsp; '
        '<span style="opacity:.7">medium (3)</span> &nbsp; '
        '<span style="opacity:1">high (4–5)</span> &nbsp;|&nbsp; '
        'Click highlighted text for details.</div>'
    )

    body = js + css + f"<h3 style='font-family:sans-serif'>{partition['student']} / {partition['lesson']} — Paragraph Error Valoration</h3>" + legend + sev_legend

    for para in partition['paragraphs']:
        pid = para['id']
        errors = errors_by_para.get(pid, [])
        sentences = para['sentences']
        score = compute_score(errors, sentences)
        slabel, scolor = score_label(score)

        # Group errors by sentence index
        errs_by_sent = defaultdict(list)
        for e in errors:
            errs_by_sent[e['sentence_index']].append(e)

        # Dimension breakdown counts
        dim_counts = defaultdict(int)
        for e in errors:
            dim_counts[e['dimension_code']] += 1

        total_words = sum(_word_count(s['text']) for s in sentences)

        body += f'<div class="para-card">'
        body += (
            f'<div class="para-header">'
            f'<span class="para-label">P{pid}. {escape(para["label"])}</span>'
            f'<span>{len(sentences)} sentences &nbsp;·&nbsp; {total_words} words &nbsp;·&nbsp; {len(errors)} errors</span>'
            f'</div>'
        )

        body += '<div class="para-body">'
        for s in sentences:
            sent_errors = errs_by_sent.get(s['index'], [])
            uid = f"p{pid}_s{s['index']}"
            body += sentence_to_html(s['text'], sent_errors, uid) + ' '
        body += '</div>'

        # Footer: score + dim breakdown
        dim_chips = ''
        for code, cnt in sorted(dim_counts.items()):
            c = DIM_COLORS[code]
            dim_chips += (f'<span style="background:{c["bg"]};border:1px solid {c["border"]};'
                          f'color:{c["badge"]};padding:1px 7px;border-radius:9px;font-weight:600;">'
                          f'{code}: {cnt}</span> ')

        body += (
            f'<div class="para-footer">'
            f'<span class="score-pill" style="background:{scolor}">Score: {score}/100</span>'
            f'<span style="color:{scolor};font-weight:600">{slabel}</span>'
            f'<span style="color:#94a3b8">|</span>'
            f'{dim_chips}'
            f'</div>'
        )
        body += '</div>'

    display(HTML(body))


render_report(partition, errors_by_para)

In [ ]:
# ── Summary table: scores per paragraph ──────────────────────────────────────

def render_summary(partition: dict, errors_by_para: dict) -> None:
    rows = ''
    for para in partition['paragraphs']:
        pid = para['id']
        errors = errors_by_para.get(pid, [])
        sentences = para['sentences']
        score = compute_score(errors, sentences)
        slabel, scolor = score_label(score)
        total_words = sum(_word_count(s['text']) for s in sentences)
        weighted_sum = sum(e['weight'] for e in errors)

        bar_w = score
        bar = (f'<div style="background:#e2e8f0;border-radius:4px;height:12px;width:120px;display:inline-block;">'
               f'<div style="background:{scolor};width:{bar_w}%;height:100%;border-radius:4px;"></div></div>')

        rows += (
            f'<tr>'
            f'<td style="padding:6px 10px;font-weight:700;">P{pid}</td>'
            f'<td style="padding:6px 10px;max-width:220px;">{escape(para["label"])}</td>'
            f'<td style="padding:6px 10px;text-align:center;">{len(sentences)}</td>'
            f'<td style="padding:6px 10px;text-align:center;">{total_words}</td>'
            f'<td style="padding:6px 10px;text-align:center;">{len(errors)}</td>'
            f'<td style="padding:6px 10px;text-align:center;">{weighted_sum}</td>'
            f'<td style="padding:6px 10px;">{bar} &nbsp;<b style="color:{scolor}">{score}</b></td>'
            f'<td style="padding:6px 10px;color:{scolor};font-weight:600">{slabel}</td>'
            f'</tr>'
        )

    html = (
        '<table style="border-collapse:collapse;font-family:sans-serif;font-size:13px;width:100%;">'
        '<thead><tr style="background:#f1f5f9;border-bottom:2px solid #cbd5e1;">'
        '<th style="padding:8px 10px;text-align:left">#</th>'
        '<th style="padding:8px 10px;text-align:left">Topic</th>'
        '<th style="padding:8px 10px">Sents</th>'
        '<th style="padding:8px 10px">Words</th>'
        '<th style="padding:8px 10px">Errors</th>'
        '<th style="padding:8px 10px">Σ Weight</th>'
        '<th style="padding:8px 10px">Score</th>'
        '<th style="padding:8px 10px">Level</th>'
        '</tr></thead>'
        f'<tbody>{rows}</tbody>'
        '</table>'
    )
    display(HTML(html))


render_summary(partition, errors_by_para)